In [ ]:
# ==========================================
# FUNÇÃO DE CHUNKING (DIVISÃO DE TEXTO)
# ==========================================
# Mesma função do notebook anterior - divide documento em seções

import re

def chunk_by_section(document_text):
    """
    Divide um documento em pedaços baseado nos títulos de seção markdown (## ).
    """
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [ ]:
# ==========================================
# CLASSE BM25INDEX - BUSCA POR PALAVRAS-CHAVE
# ==========================================

import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    """
    BM25 (Best Matching 25) - Algoritmo de busca baseado em PALAVRAS-CHAVE.
    
    🔍 DIFERENÇA FUNDAMENTAL DO VECTOR SEARCH:
    - Vector Search: entende SIGNIFICADO semântico ("carro" ≈ "automóvel")
    - BM25: procura PALAVRAS EXATAS ("carro" ≠ "automóvel")
    
    💡 QUANDO USAR BM25?
    - Buscas por nomes específicos (pessoas, produtos, códigos)
    - Termos técnicos exatos
    - Combinar com vector search para busca híbrida
    
    🎯 COMO FUNCIONA?
    1. Tokeniza texto em palavras (tokenization)
    2. Calcula frequência dos termos (TF - Term Frequency)
    3. Calcula raridade dos termos (IDF - Inverse Document Frequency)
    4. Combina TF e IDF com penalização por tamanho do documento
    
    PARÂMETROS IMPORTANTES:
    - k1: controla saturação da frequência de termos (padrão: 1.5)
          -> valores altos: mais peso para repetições da palavra
    - b: controla penalização por tamanho do documento (padrão: 0.75)
          -> 0 = sem penalização, 1 = penalização máxima
    """
    
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        """
        Inicializa o índice BM25.
        
        ESTRUTURAS DE DADOS:
        - documents: lista de documentos originais
        - _corpus_tokens: documentos tokenizados (palavras separadas)
        - _doc_len: comprimento de cada documento em tokens
        - _doc_freqs: quantos documentos contêm cada termo
        - _avg_doc_len: tamanho médio dos documentos
        - _idf: score IDF (raridade) de cada termo
        - _index_built: flag indicando se o índice foi construído
        """
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        """
        Tokenizador padrão: converte texto em lista de palavras.
        
        PROCESSO:
        1. Converte para minúsculas ("Python" → "python")
        2. Divide por caracteres não-alfanuméricos (split por espaços, pontuação)
        3. Remove tokens vazios
        
        EXEMPLO:
        "Hello, World! 123" → ["hello", "world", "123"]
        """
        text = text.lower()
        tokens = re.split(r"\W+", text)  # \W+ = um ou mais caracteres não-alfanuméricos
        return [token for token in tokens if token]  # Remove strings vazias

    def _update_stats_add(self, doc_tokens: List[str]):
        """
        Atualiza estatísticas ao adicionar um novo documento.
        
        CALCULA:
        1. Tamanho do documento (número de tokens)
        2. Document Frequency (DF): em quantos documentos cada termo aparece
        
        POR QUE DF É IMPORTANTE?
        - Termos raros (baixo DF) são mais informativos
        - Termos comuns (alto DF) são menos relevantes
        - "engenharia" (raro) > "o", "a", "de" (comum)
        """
        self._doc_len.append(len(doc_tokens))

        # Conta cada termo apenas UMA vez por documento
        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False  # Força reconstrução do índice

    def _calculate_idf(self):
        """
        Calcula IDF (Inverse Document Frequency) para cada termo.
        
        📊 O QUE É IDF?
        - Mede a RARIDADE de um termo no corpus
        - Termos raros têm IDF ALTO → mais importantes
        - Termos comuns têm IDF BAIXO → menos importantes
        
        FÓRMULA BM25 IDF:
        IDF(termo) = log((N - df + 0.5) / (df + 0.5) + 1)
        
        Onde:
        - N = número total de documentos
        - df = document frequency (em quantos docs o termo aparece)
        - +0.5 e +1 são ajustes para evitar valores negativos
        
        EXEMPLO PRÁTICO:
        Se temos 100 documentos:
        - "python" aparece em 80 docs → IDF baixo (comum)
        - "kubernetes" aparece em 5 docs → IDF alto (raro)
        """
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        """
        Constrói/reconstrói o índice BM25.
        
        ETAPAS:
        1. Calcula tamanho médio dos documentos
        2. Calcula IDF de todos os termos
        3. Marca índice como construído
        
        POR QUE TAMANHO MÉDIO?
        - Documentos longos naturalmente têm mais termos
        - BM25 normaliza por tamanho para ser justo
        - Doc pequeno com "python" 2x > Doc gigante com "python" 2x
        """
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        """
        Adiciona um documento ao índice.
        
        VALIDAÇÕES:
        - Documento deve ser dicionário
        - Deve ter chave 'content' com texto
        
        PROCESSO:
        1. Tokeniza o conteúdo
        2. Armazena documento e tokens
        3. Atualiza estatísticas
        """
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        """
        Calcula o score BM25 entre query e um documento.
        
        🎯 FÓRMULA BM25 (SIMPLIFICADA):
        Para cada termo da query:
            Score += IDF(termo) × (TF × (k1 + 1)) / (TF + k1 × (1 - b + b × (DL / AvgDL)))
        
        COMPONENTES:
        - IDF: raridade do termo (calculado antes)
        - TF: quantas vezes o termo aparece no documento
        - DL: tamanho do documento atual
        - AvgDL: tamanho médio dos documentos
        - k1: controla saturação (quanto mais repetições importam)
        - b: controla penalização por tamanho
        
        💡 INTUIÇÃO:
        - Termo raro (alto IDF) + aparece muito (alto TF) = Score ALTO
        - Termo comum (baixo IDF) = Score BAIXO (mesmo com alto TF)
        - Documento muito longo = penalizado (evita favorecer docs grandes)
        
        EXEMPLO:
        Query: "python machine learning"
        Doc A: "python python python" (3x python, 0x learning) → score ~médio
        Doc B: "python machine learning tutorial" (1x cada) → score ~alto
        """
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])  # Conta frequência de cada termo
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue  # Termo não existe no corpus, ignora

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)  # TF: quantas vezes aparece

            # Fórmula BM25
            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)  # +1e-9 evita divisão por zero

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        """
        Busca os k documentos mais relevantes para a query.
        
        FLUXO:
        1. Valida entrada
        2. Constrói índice se necessário
        3. Tokeniza a query
        4. Calcula score BM25 para cada documento
        5. Ordena por score (maior = mais relevante)
        6. Normaliza scores para formato similar ao Vector Search
        7. Retorna top-k resultados
        
        NORMALIZAÇÃO DE SCORE:
        - BM25 retorna scores ALTOS para relevância ALTA
        - Vector Search retorna distâncias BAIXAS para relevância ALTA
        - Usamos exp(-factor × score) para converter:
          * Score alto → distância baixa
          * Score baixo → distância alta
        - Isso facilita combinar BM25 com Vector Search depois!
        
        RETORNA: Lista de tuplas (documento, distância_normalizada)
        - Distância MENOR = mais relevante (alinhado com Vector Search)
        """
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        # Constrói índice se ainda não foi construído
        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        # Tokeniza a query (mesma forma que os documentos)
        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        # Calcula score para todos os documentos
        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:  # Ignora scores muito baixos
                raw_scores.append((raw_score, self.documents[i]))

        # Ordena por score (MAIOR primeiro = mais relevante)
        raw_scores.sort(key=lambda item: item[0], reverse=True)

        # Normaliza scores: converte BM25 score alto → distância baixa
        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        # Ordena por distância (MENOR primeiro, para consistência com Vector Search)
        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25Index(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [ ]:
# ==========================================
# CARREGAR O DOCUMENTO
# ==========================================
# Mesmo arquivo report.md usado no notebook de Vector Search

with open("./report.md", "r") as f:
    text = f.read()

In [ ]:
# ==========================================
# PASSO 1: DIVIDIR O TEXTO EM CHUNKS
# ==========================================
# Divide o documento em seções para indexação

chunks = chunk_by_section(text)
print(f"Total de chunks criados: {len(chunks)}")
print(f"\nExemplo do 1º chunk (primeiros 200 caracteres):")
print(chunks[0][:200] + "...")

In [ ]:
# ==========================================
# PASSO 2: CRIAR ÍNDICE BM25 E ADICIONAR CHUNKS
# ==========================================
# Cria um índice BM25 e adiciona cada chunk
# Cada chunk será tokenizado e indexado para busca por palavras-chave

bm25_store = BM25Index()

for chunk in chunks:
    bm25_store.add_document({"content": chunk})

print(f"Índice BM25 criado: {bm25_store}")
print(f"Total de documentos indexados: {len(bm25_store)}")

In [ ]:
# ==========================================
# PASSO 3: BUSCAR NO ÍNDICE BM25
# ==========================================
# Busca os 2 chunks mais relevantes usando BM25 (busca por palavras-chave)
# 
# QUERY: "software engineering department"
# 
# COMO BM25 BUSCA:
# 1. Tokeniza a query: ["software", "engineering", "department"]
# 2. Para cada documento, calcula:
#    - Quantas vezes cada palavra aparece (TF)
#    - Quão rara é cada palavra no corpus (IDF)
#    - Ajusta pelo tamanho do documento
# 3. Retorna documentos com maior score (mais palavras correspondentes)
#
# DIFERENÇA DO VECTOR SEARCH:
# - BM25: procura palavras EXATAS → encontra docs com "software", "engineering", "department"
# - Vector: procura SIGNIFICADO → encontra docs sobre "desenvolvimento", "equipe técnica", etc.

results = bm25_store.search("software engineering department", k=2)

print("=" * 60)
print("RESULTADOS DA BUSCA BM25")
print("=" * 60)

for i, (doc, distance) in enumerate(results, 1):
    print(f"\n📄 RESULTADO #{i}")
    print(f"Distância normalizada: {distance:.4f}  (menor = mais relevante)")
    print(f"\nConteúdo (primeiros 300 caracteres):")
    print(doc['content'][:300] + "...")
    print("-" * 60)